# 04 · Query-based: RT-DETR-L (Ultralytics)
Same data copy, frame step, epochs and evaluator as the YOLO runs → comparable compute budget.
Ultralytics' YOLO and RT-DETR log the same per-epoch val metric, so their curves can be plotted together.

Kaggle: Accelerator **GPU**, **Internet on**. Run with *Save Version → Save & Run All*.

## 0. Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/YOUR_USERNAME/YOUR_REPO.git'  # <- change once: your public repo with src/common.py
FAST_DEV = os.environ.get('FAST_DEV') == '1'                # tiny smoke test on a fake dataset; keep off on Kaggle

if not any(Path(d, 'common.py').exists() for d in ['src', '../src', 'repo/src']):
    subprocess.run(['git', 'clone', '-q', REPO_URL, 'repo'], check=True)
SRC = next(d for d in ['src', '../src', 'repo/src'] if Path(d, 'common.py').exists())
sys.path.insert(0, SRC)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.208', 'pycocotools'], check=True)

import pickle
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import ultralytics
from ultralytics import RTDETR
from common import *

seed_everything()
ROOT = find_data_root()
TMP = Path('/tmp/hw2') if not FAST_DEV else Path('work_dev')
OUT = Path('results'); OUT.mkdir(exist_ok=True)
FIG = Path('images'); FIG.mkdir(exist_ok=True)
WEIGHTS = Path('weights'); WEIGHTS.mkdir(exist_ok=True)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 1. Protocol (same as 03)

In [ ]:
STEP = 5
EPOCHS = 15 if not FAST_DEV else 1
yaml_path = prepare_training_copy(ROOT, TMP / 'data', step_train=STEP, step_val=STEP)
val_frames = list_frames(ROOT / 'val', STEP)
evaluator = CocoEvaluator(ROOT, val_frames)

COMMON_ARGS = dict(data=str(yaml_path), epochs=EPOCHS, seed=SEED, deterministic=False, cos_lr=True, close_mosaic=3,
                   workers=4, project=str((TMP / 'runs').resolve()), exist_ok=True, plots=True, verbose=False,
                   optimizer='AdamW', lr0=1e-4, weight_decay=1e-4, warmup_epochs=1,
                   device=0 if torch.cuda.is_available() else 'cpu')

## 2. Number of queries vs. scene density
RT-DETR keeps the top-K encoder tokens as decoder queries (K = 300 by default). Each query outputs at most one
object, so frames with more objects than queries cannot be fully detected. `00_eda` shows how many frames exceed 300.
Below we create a config with K = 500 (pretrained weights still load: no parameter depends on K).

In [ ]:
base_yaml = Path(ultralytics.__file__).parent / 'cfg/models/rt-detr/rtdetr-l.yaml'
txt = base_yaml.read_text()
assert 'RTDETRDecoder, [nc]]' in txt
q500_yaml = TMP / 'rtdetr-l-q500.yaml'
q500_yaml.parent.mkdir(parents=True, exist_ok=True)
q500_yaml.write_text(txt.replace('RTDETRDecoder, [nc]]', 'RTDETRDecoder, [nc, 256, 500]]'))  # args: nc, hidden dim, num queries

## 3. Experiments
* `rtdetr_640` – RT-DETR-L, 300 queries, imgsz 640 (baseline, same resolution as `yolo_640`).
* `rtdetr_q500_640` – 500 queries (dense scenes).
* `rtdetr_q500_1024` – 500 queries, imgsz 1024 (optional, slow).
Optimizer: AdamW lr 1e-4 (transformer detectors diverge with the SGD lr 0.01 Ultralytics would pick).

In [ ]:
EXPERIMENTS = {
    'rtdetr_640':       dict(cfg='rtdetr-l.yaml', imgsz=640, batch=8),
    'rtdetr_q500_640':  dict(cfg=str(q500_yaml), imgsz=640, batch=8),
    'rtdetr_q500_1024': dict(cfg=str(q500_yaml), imgsz=1024, batch=4),
}
RUN = ['rtdetr_640', 'rtdetr_q500_640']  # add 'rtdetr_q500_1024' if GPU time allows
if FAST_DEV:
    EXPERIMENTS = {k: dict(v, imgsz=320, batch=2) for k, v in EXPERIMENTS.items()}

## 4. Train and evaluate

In [ ]:
results = {}
for name in RUN:
    cfg = EXPERIMENTS[name]
    print(f'\n===== {name} =====')
    seed_everything()
    model = RTDETR(cfg['cfg'])
    model.load('rtdetr-l.pt')  # COCO-pretrained weights
    model.train(name=name, imgsz=cfg['imgsz'], batch=cfg['batch'], **COMMON_ARGS)
    run_dir = Path(model.trainer.save_dir)
    best = run_dir / 'weights' / 'best.pt'
    shutil.copy(best, WEIGHTS / f'{name}.pt')
    shutil.copy(run_dir / 'results.csv', OUT / f'ultra_{name}.csv')
    model = RTDETR(best)
    preds, fps = predict_ultralytics(model, val_frames, imgsz=cfg['imgsz'])
    m = evaluator.evaluate(preds)
    results[name] = preds
    log_result(dict(model=name, family='query-based', epochs=EPOCHS, step=STEP, imgsz=cfg['imgsz'], fps=fps, **m),
               OUT / 'metrics.csv')
    pickle.dump(preds, open(TMP / f'val_preds_{name}.pkl', 'wb'))
    print(name, {k: m[k] for k in ['mAP', 'AP50', 'AP_small']}, 'fps', fps)
    del model; torch.cuda.empty_cache()

## 5. Results

In [ ]:
tab = pd.read_csv(OUT / 'metrics.csv')
cols = ['model', 'family', 'mAP', 'AP50', 'AP75', 'AP_small', 'AP_medium', 'AR500', 'fps']
print(tab[cols].to_string(index=False))
size_tab = pd.concat([evaluator.evaluate_by_size(p).assign(model=n) for n, p in results.items()])
size_tab.to_csv(OUT / 'size_breakdown_rtdetr.csv', index=False)
print(size_tab.pivot(index='size', columns='model', values='AP50'))

## 6. Same budget: RT-DETR vs YOLO (CNN) — val mAP over training
Needs `results/ultra_yolo_640.csv` from notebook 03 (committed to the repo, or in this session's results/).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
curves = {n: OUT / f'ultra_{n}.csv' for n in RUN}
for cand in [OUT / 'ultra_yolo_640.csv', Path(SRC).parent / 'results' / 'ultra_yolo_640.csv']:
    if cand.exists():
        curves['yolo_640 (CNN)'] = cand
        break
for n, path in curves.items():
    h = pd.read_csv(path); h.columns = h.columns.str.strip()
    ax[0].plot(h.epoch, h['metrics/mAP50-95(B)'], marker='.', label=n)
    if 'time' in h:
        ax[1].plot(h['time'] / 60, h['metrics/mAP50-95(B)'], marker='.', label=n)
ax[0].set_xlabel('epoch'); ax[1].set_xlabel('training time, min'); ax[0].set_ylabel('val mAP50-95')
ax[0].legend(); ax[1].legend(); fig.suptitle('Query-based vs CNN under the same budget')
plt.tight_layout(); plt.savefig(FIG / 'detr_vs_yolo.png', dpi=120); plt.show()

## 7. Dense scenes: recall vs. number of objects in the frame

In [ ]:
BEST = max(results, key=lambda n: evaluator.evaluate(results[n])['mAP'])
err = frame_errors(ROOT, val_frames, results[BEST], score_thr=0.3)
err['density'] = pd.cut(err.n_gt, [-1, 25, 50, 100, 200, 10000])
print(err.groupby('density', observed=True).apply(lambda d: pd.Series({'frames': len(d), 'recall@0.3': d.tp.sum() / max(1, d.n_gt.sum())})).round(3))

## 8. Notes for the report
*(your analysis: bipartite (Hungarian) matching → one prediction per object → no NMS; slower convergence;
has it converged in the budget? small objects and dense scenes vs YOLO)*